# MNIST: TensorFlow graph và Session / Graph-mode neural network

**Người thực hành / Learner:** Trương Minh Hoàng · **MSSV / Student ID:** 25410008

Bài thực hành để đọc, chỉnh sửa và giải thích quy trình học sâu. / A learning exercise in understanding and improving a deep-learning pipeline.

**Nguồn / Provenance:** Bản Colab tham khảo có tên gốc `HW_HaMinhTan_03.ipynb`. Thông tin người thực hành ở trên được chuẩn hóa cho portfolio; tên nguồn được giữ để phân biệt việc thực hành với tác giả ban đầu.

**Input → Output:** ảnh MNIST → nhãn chữ số. Tôi luyện placeholder, biến, đồ thị tính toán, feed_dict và Session, rồi đối chiếu với cách dùng Keras hiện đại. / Practice explicit graph construction and session-based optimization.

In [1]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [2]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
tf.reset_default_graph()
tf.set_random_seed(42)

from tensorflow.keras.datasets import mnist
import numpy as np

# Để kết quả ổn định hơn
tf.set_random_seed(42)
np.random.seed(42)

# =========================
# Hyperparameters
# =========================
learningRate = 0.001
batchSize = 50
NumberUnit = 100
epoches = 10

print("This system has batch size =", batchSize,
      "| Number of units =", NumberUnit,
      "| learning rate =", learningRate)

# =========================
# Load and preprocess MNIST
# =========================
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Flatten 28x28 -> 784 và scale về [0,1]
X_train = X_train.reshape(-1, 784).astype(np.float32) / 255.0
X_test = X_test.reshape(-1, 784).astype(np.float32) / 255.0
y_train = y_train.astype(np.int64)
y_test = y_test.astype(np.int64)

from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)
num_examples = X_train.shape[0]
total_batch = num_examples // batchSize

# =========================
# Model
# =========================
def fully_connected(x, units, activation_fn=None, scope="fc"):
    with tf.variable_scope(scope):
        in_dim = int(x.get_shape()[1])
        W = tf.get_variable(
            "weights",
            shape=[in_dim, units],
            initializer=tf.glorot_uniform_initializer()
        )
        b = tf.get_variable(
            "biases",
            shape=[units],
            initializer=tf.zeros_initializer()
        )
        z = tf.matmul(x, W) + b
        if activation_fn is not None:
            z = activation_fn(z)
        return z

def deepNeuronNetwork(X, NumberUnit):
    h1 = fully_connected(X, NumberUnit, activation_fn=tf.nn.relu, scope="h1")
    h2 = fully_connected(h1, NumberUnit, activation_fn=tf.nn.relu, scope="h2")
    h3 = fully_connected(h2, NumberUnit, activation_fn=tf.nn.relu, scope="h3")
    out = fully_connected(h3, 10, activation_fn=None, scope="out")
    return out

# =========================
# Graph
# =========================
X = tf.placeholder(tf.float32, [None, 784], name="X")
y = tf.placeholder(tf.int64, [None], name="y")

logits = deepNeuronNetwork(X, NumberUnit)

entropy = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=logits)
loss = tf.reduce_mean(entropy, name="loss")

optimizer = tf.train.AdamOptimizer(learning_rate=learningRate)
training_op = optimizer.minimize(loss)

correct = tf.nn.in_top_k(predictions=logits, targets=y, k=1)
accuracy = tf.reduce_mean(tf.cast(correct, tf.float32))

init = tf.global_variables_initializer()

# =========================
# Training
# =========================
with tf.Session() as sess:
    sess.run(init)

    for e in range(epoches):
        # shuffle mỗi epoch
        indices = np.random.permutation(num_examples)
        X_train_shuffled = X_train[indices]
        y_train_shuffled = y_train[indices]

        avg_acc = 0.0
        avg_loss = 0.0

        for iteration in range(total_batch):
            start = iteration * batchSize
            end = start + batchSize

            X_batch = X_train_shuffled[start:end]
            y_batch = y_train_shuffled[start:end]

            _, batch_acc, batch_loss = sess.run(
                [training_op, accuracy, loss],
                feed_dict={X: X_batch, y: y_batch}
            )

            avg_acc += batch_acc / total_batch
            avg_loss += batch_loss / total_batch

        trainAcc = sess.run(accuracy, feed_dict={X: X_train, y: y_train})
        validAcc = sess.run(accuracy, feed_dict={X: X_valid, y: y_valid})

        print(
            "Epoch:", "%03d" % (e + 1),
            "| Train accuracy:", round(float(trainAcc), 6),
            "| Avg batch acc:", round(float(avg_acc), 6),
            "| Loss:", round(float(avg_loss), 6),
            "| Validation accuracy:", round(float(validAcc), 6)
        )
    final_test_accuracy = float(sess.run(accuracy, feed_dict={X: X_test, y: y_test}))
    LAB_RESULT = {"test_accuracy": final_test_accuracy, "epochs": epoches, "train_rows": len(y_train), "validation_rows": len(y_valid), "test_rows": len(y_test)}
    print("Final test:", LAB_RESULT)


Instructions for updating:
non-resource variables are not supported in the long term


This system has batch size = 50 | Number of units = 100 | learning rate = 0.001


Epoch: 001 | Train accuracy: 0.962537 | Avg batch acc: 0.920091 | Loss: 0.269335 | Validation accuracy: 0.952333


Epoch: 002 | Train accuracy: 0.973241 | Avg batch acc: 0.965073 | Loss: 0.113518 | Validation accuracy: 0.962333


Epoch: 003 | Train accuracy: 0.982037 | Avg batch acc: 0.974479 | Loss: 0.081455 | Validation accuracy: 0.969


...

Epoch: 006 | Train accuracy: 0.992481 | Avg batch acc: 0.985385 | Loss: 0.044115 | Validation accuracy: 0.974833


Epoch: 007 | Train accuracy: 0.993537 | Avg batch acc: 0.988422 | Loss: 0.035534 | Validation accuracy: 0.974333


Epoch: 008 | Train accuracy: 0.990315 | Avg batch acc: 0.98968 | Loss: 0.030978 | Validation accuracy: 0.9705


Epoch: 009 | Train accuracy: 0.994556 | Avg batch acc: 0.991069 | Loss: 0.026296 | Validation accuracy: 0.975833


Epoch: 010 | Train accuracy: 0.992074 | Avg batch acc: 0.992364 | Loss: 0.023921 | Validation accuracy: 0.973167
Final test: {'test_accuracy': 0.9745000004768372, 'epochs': 10, 'train_rows': 54000, 'validation_rows': 6000, 'test_rows': 10000}


## Điều tôi luyện thêm / Additional practice

Đọc sự khác nhau giữa TensorFlow 1 graph/session và TensorFlow 2 eager/Keras. Tôi tách validation từ train, theo dõi validation trong mỗi epoch và chỉ đánh giá test sau khi huấn luyện xong. Chạy bài này trong kernel riêng vì `disable_v2_behavior()` thay đổi chế độ TensorFlow của cả tiến trình.